In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
import pandas as pd
import torch, torchvision
print(torch.__version__, torch.version.cuda)
print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

2.14.1+cu126 12.6
0.29.1+cu126
CUDA available: True


In [2]:
#importing dataset

df = pd.read_csv("../datasets/pubmed abstracts/pubmed_abstracts copy.csv")
df.head()

,Unnamed: 0,deep_learning
0,0,(['Magnetic resonance spectroscopic imaging (M...
1,1,(['Existing deep convolutional neural networks...
2,2,(['Deep learning techniques have been increasi...
3,3,(['The original article unfortunately containe...
4,4,(['The most common applications of artificial ...


In [3]:
#loading dataset in langchain

docs = []
for col in df.columns:
    if col == "Unnamed: 0":
        continue
    text = "\n".join(df[col].dropna().astype(str))
    docs.append(Document(page_content=text, metadata={"column": col}))

print(len(docs))   # = number of columns
print(docs[0].metadata)
print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

22842
by the classification branch. Feeding a chest radiograph to the trained Thorax-Net, a diagnosis is obtained by averaging and binarizing the outputs of two branches. The proposed Thorax-Net model has been evaluated against three state-of-the-art deep learning models using the patientwise official split of the ChestX-ray14 dataset and against other five deep learning models using the imagewise random data split. Our results show that Thorax-Net achieves an average per-class area under the receiver operating characteristic curve (AUC) of 0.7876 and 0.896 in both experiments, respectively, which are higher than the AUC values obtained by other deep models when they were all trained with no external data.'], 'Thorax-Net: An Attention Regularized Deep Neural Network for Classification of Thoracic Diseases on Chest Radiography.')


In [5]:
#Creating Embaddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 8},
    )

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1005.58it/s]


In [6]:
#Vector database

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(texts)

# torch.cuda.synchronize()
# print("Embedding dimensions:", len(query_embedding))
# print("GPU memory in use:", round(torch.cuda.memory_allocated() / 1024**2, 1), "MiB")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "s:\Projects\rag-dissertation\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\ankit\AppData\Local\Temp\ipykernel_16104\1093081040.py", line 4, in <module>
    vector_store.add_documents(texts)
  File "s:\Projects\rag-dissertation\.venv\Lib\site-packages\langchain_core\vectorstores\in_memory.py", line 195, in add_documents
    vectors = self.embedding.embed_documents(texts)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "s:\Projects\rag-dissertation\.venv\Lib\site-packages\langchain_huggingface\embeddings\huggingface.py", line 137, in embed_documents
    return self._embed(texts, self.encode_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "s:\Projects\rag-dissertation\.venv\Lib\site-packages\langchain_huggingface\embeddings\huggingface.py", line 112, in _embed
    embeddings = self._client.encode(
                

In [ ]:
#Setting up LLM

llm = ChatOllama(
    model="gemma4:e4b", 
    )

In [ ]:
#Query LLM

response = llm.invoke("Hey how u doing today")

In [6]:
print(response.text)

I'm doing great, thank you for asking! 😊 Just here and ready to chat.

How about you? What's going on with you today?
